# 003 — Рыночные режимы: параметры ↔ режим

Объединяет `parameter_performance_by_slice` (AN-1) с `time_slice_features` (REG-2)
и считает корреляции параметров с фичами режима + лучшие параметры по режиму
(REG-3). При отсутствии реальных таблиц используется синтетика — ноутбук
исполняется сверху донизу без правок.

In [ ]:
import os
import tempfile
from datetime import UTC, datetime
from pathlib import Path

import polars as pl

from trading_research.analytics.correlations import (
    attach_regime_features,
    best_params_in_regime,
    regime_correlations,
    regime_metric_summary,
)
from trading_research.analytics.queries import (
    PerformanceRecord,
    build_parameter_performance,
    load_performance_table,
)
from trading_research.backtest.metrics import Metrics
from trading_research.domain import TimeSlice, TimeSliceKind

In [ ]:
SLICES = ["2024-01", "2024-02", "2024-03", "2024-04"]
P1, P2 = {"fast_ma": 5, "slow_ma": 20}, {"fast_ma": 10, "slow_ma": 50}


def _mk(total):
    return Metrics(total, -5.0, 1.0, 1.0, 2.0, 55.0, 0.3, 1.1, 2.0, 7, 8760.0, 30, True)


def _sl(sid):
    m = int(sid.split("-")[1])
    return TimeSlice(
        id=sid,
        start=datetime(2024, m, 1, tzinfo=UTC),
        end=datetime(2024, m + 1, 1, tzinfo=UTC),
        label=sid,
        kind=TimeSliceKind.MONTH,
    )


def _synthetic_performance():
    recs = []
    for params, totals in [(P1, [10, 20, 30, 40]), (P2, [40, 30, 20, 10])]:
        for sid, t in zip(SLICES, totals, strict=True):
            recs.append(
                PerformanceRecord(
                    "demo", "ma_cross", "BTCUSDT", "1h", _sl(sid), params, _mk(float(t))
                )
            )
    return build_parameter_performance(recs)


def _synthetic_features():
    return pl.DataFrame(
        {
            "symbol": ["BTCUSDT"] * 4,
            "timeframe": ["1h"] * 4,
            "time_slice_id": SLICES,
            "market_return_pct": [1.0, 2.0, 3.0, 4.0],
            "volatility": [1.0, 2.0, 3.0, 4.0],
            "trend_strength": [4.0, 3.0, 2.0, 1.0],
            "funding_avg": [1e-4, 2e-4, 3e-4, 4e-4],
            "oi_change_pct": [1.0, 1.0, 1.0, 1.0],
            "ls_ratio_avg": [1.5, 1.5, 1.5, 1.5],
            "regime_label": ["sideways_low_vol", "sideways_high_vol", "bull_trend", "bull_trend"],
        }
    )


perf_path = os.environ.get("TR_PERFORMANCE")
feat_path = os.environ.get("TR_FEATURES")
perf = (
    load_performance_table(perf_path)
    if perf_path and Path(perf_path).exists()
    else _synthetic_performance()
)
features = (
    pl.read_parquet(feat_path) if feat_path and Path(feat_path).exists() else _synthetic_features()
)
joined = attach_regime_features(perf, features)
joined.head()

## Корреляции: параметр ↔ фичи режима

In [ ]:
corr = regime_correlations(joined, metric="total_return_pct")
corr

## Средняя доходность по режимам (вопросы раздела 9)

In [ ]:
summary = regime_metric_summary(joined, metric="total_return_pct")
summary

In [ ]:
# Какие параметры лучше в конкретном режиме (напр. bull_trend / sideways_high_vol).
best = best_params_in_regime(joined, regime_label="bull_trend", metric="total_return_pct")
best

## График корреляций (bar)

In [ ]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

feat_cols = [c for c in corr.columns if c != "param_hash"]
row = corr.row(0, named=True)
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(feat_cols, [row[c] if row[c] is not None else 0.0 for c in feat_cols])
ax.set_ylabel("corr(total_return_pct, feature)")
ax.set_title(f"param {row['param_hash']}")
ax.tick_params(axis="x", rotation=45)
out = Path(tempfile.mkdtemp()) / "regime_corr.png"
fig.tight_layout()
fig.savefig(out)
plt.close(fig)
print("saved", out)